In [2]:
import os
import faiss
from dotenv import load_dotenv
from transformers import AutoTokenizer

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    set_global_tokenizer,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.vector_stores.faiss import FaissVectorStore

In [3]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [4]:
set_global_tokenizer(AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5").encode)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.chunk_size = 450
Settings.chunk_overlap = 50

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
documents = SimpleDirectoryReader(
    input_dir="/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data",
    exclude=[
        "/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/README.md",
        "/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/paul_graham_essay.txt",
    ],
).load_data(num_workers=4)

In [6]:
print(f"Loaded {len(documents)} documents.")

Loaded 11 documents.


In [7]:
documents[0].metadata

{'file_path': '/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/10min.rst',
 'file_name': '10min.rst',
 'file_type': 'text/x-rst',
 'file_size': 18933,
 'creation_date': '2026-07-21',
 'last_modified_date': '2026-07-21'}

In [8]:
d = 384
faiss_index = faiss.IndexFlatL2(d)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [9]:
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, show_progress=True
)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (6682 > 512). Running this sequence through the model will result in indexing errors


Generating embeddings:   0%|          | 0/717 [00:00<?, ?it/s]

In [10]:
print(len(index.docstore.docs))

717


In [11]:
for node_id, node in list(index.docstore.docs.items())[:5]:
    print("--- CHUNK ---")
    print("Source file:", node.metadata.get("file_name"))
    print("Text preview:", node.text[:300])
    print()

--- CHUNK ---
Source file: 10min.rst
Text preview: .. _10min:

{{ header }}

********************
10 minutes to pandas
********************

This is a short introduction to pandas, geared mainly for new users.
You can see more complex recipes in the :ref:`Cookbook<cookbook>`.

Customarily, we import as follows:

.. ipython:: python

   import numpy 

--- CHUNK ---
Source file: 10min.rst
Text preview: Creating a :class:`Series` by passing a list of values, letting pandas create
a default :class:`RangeIndex`.

.. ipython:: python

   s = pd.Series([1, 3, 5, np.nan, 6, 8])
   s

Creating a :class:`DataFrame` by passing a NumPy array with a datetime index using :func:`date_range`
and labeled columns

--- CHUNK ---
Source file: 10min.rst
Text preview: Here's a subset of the attributes that
will be completed:

.. ipython::

   @verbatim
   In [1]: df2.<TAB>  # noqa: E225, E999
   df2.A                  df2.bool
   df2.abs                df2.boxplot
   df2.add                df2.C
   df2.add_

In [12]:
for node_id, node in list(index.docstore.docs.items())[:5]:
    print(len(node.text), "characters")

918 characters
1200 characters
804 characters
879 characters
1004 characters


In [14]:
for node_id, node in list(index.docstore.docs.items())[:12]:
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")
    print(len(tok.encode(node.text)), "tokens")

297 tokens
398 tokens
263 tokens
285 tokens
325 tokens
302 tokens
259 tokens
242 tokens
408 tokens
270 tokens
352 tokens
384 tokens


In [15]:
print(index.ref_doc_info)

{'c29961e6-f56e-4111-94a0-c33303f93651': RefDocInfo(node_ids=['27ab7a49-5910-4f80-921c-340ff999f311', '0b21545e-cab5-4219-9131-bea4de202113', '35a5c0db-c4a5-4b51-990b-a398f279f3c3', 'db1f4b81-a380-44b4-9293-009fccea1063', '059eb914-2ab5-425b-b041-f39a40e979e6', 'c0b73034-9db7-4b1a-96ab-4371f035c7a5', '741443b5-066e-47d6-b6be-877e64cfa15e', '82fc9109-3746-42c1-b8ca-6c8edf2bfaa2', '670b65e8-fb84-4835-81b4-c91c1b6f368c', 'db2be59a-c913-4aa1-ba13-7cde1c1a8a50', 'd528f3c3-200f-4299-9014-49d6cc3f08e8', 'bba4eedf-2950-4dc7-9a7b-f7bfc8b4f9e8', 'd7ce86e9-e0f5-4a13-8178-f2d934b2dca9', '946ed95b-0e1c-4574-82c4-6d9adddbafa5', '88d9be96-c334-468e-a52d-1dc669878daf', 'c8c93849-9726-46d4-88c0-3ce5155c81df', '245b8386-fa63-49cc-a03e-390b83f0b03f', '7fc658ea-78f8-4d7f-a974-cf124309f193', '9b4c9c9b-d703-467c-a93b-8944aebb4ada', 'a2d52054-979b-4a30-a7e4-264eb4ea6559', 'ae88d426-1b07-4f48-97d5-34f6d02e0744', '8a67d05b-fa25-4d18-befb-191e09e47467'], metadata={'file_path': '/Users/mohitag/Documents/Projects

In [18]:
print(documents[0].text)

.. _10min:

{{ header }}

********************
10 minutes to pandas
********************

This is a short introduction to pandas, geared mainly for new users.
You can see more complex recipes in the :ref:`Cookbook<cookbook>`.

Customarily, we import as follows:

.. ipython:: python

   import numpy as np
   import pandas as pd

Basic data structures in pandas
-------------------------------

pandas provides two types of classes for handling data:

1. :class:`Series`: a one-dimensional labeled array holding data of any type
    such as integers, strings, Python objects etc.
2. :class:`DataFrame`: a two-dimensional data structure that holds data like
   a two-dimension array or a table with rows and columns.

Object creation
---------------

See the :ref:`Intro to data structures section <dsintro>`.

Creating a :class:`Series` by passing a list of values, letting pandas create
a default :class:`RangeIndex`.

.. ipython:: python

   s = pd.Series([1, 3, 5, np.nan, 6, 8])
   s

Creating a 

In [20]:
Settings.llm = HuggingFaceInferenceAPI(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    token=hf_token,
    temperature=0.2,
    max_tokens=256,
    provider="auto",
)

In [21]:
query_engine = index.as_query_engine()

In [22]:
response = query_engine.query("how to create a csv file?")

In [24]:
print(response.response)

You can create a CSV file by using the `to_csv` method of a `Series` or `DataFrame` object. This method allows you to store the contents of the object as a comma-separated-values file. The `to_csv` method takes a number of arguments, but only the first one is required.


In [25]:
response = query_engine.query("create a dataframe?")

In [26]:
print(response.response)

Here is a simple example of creating a DataFrame:

```python
import pandas as pd
import numpy as np

data = {
    "Name": ["John", "Anna", "Peter", "Linda"],
    "Age": [28, 24, 35, 32],
    "Country": ["USA", "UK", "Australia", "Germany"]
}

df = pd.DataFrame(data)
print(df)
```
